In [1]:
from pytrends.request import TrendReq
import pandas as pd
from datetime import datetime, timedelta
import time
import random

In [2]:
ANCHOR = 'google'
KEYWORDS = ['gimnasio', 'dieta', ANCHOR]
GEO = 'PE'

pytrends = TrendReq(hl='es-PE', tz=300)

In [3]:

def fetch_daily_trends(keywords, start, end,
                       geo='PE', window_days=30,
                       sleep_range=(20, 35), max_retries=2):
    all_data = []
    current_start = start

    while current_start < end:
        current_end = min(current_start + timedelta(days=window_days), end)

        attempt = 0
        while attempt < max_retries:
            try:
                pytrends.build_payload(
                    keywords,
                    timeframe=f"{current_start.date()} {current_end.date()}",
                    geo=geo
                )

                df = pytrends.interest_over_time()
                if not df.empty:
                    all_data.append(df.reset_index())

                time.sleep(random.uniform(*sleep_range))
                break

            except Exception as e:
                attempt += 1
                wait = 15 * attempt
                print(f"⚠️ Retry {attempt} ({current_start.date()}): {e}")
                time.sleep(wait)

        current_start = current_end

    return pd.concat(all_data, ignore_index=True)

In [ ]:
start = datetime(2019, 1, 1)
end = datetime(2026, 1, 2)

df = fetch_daily_trends(KEYWORDS, start, end)
df = df.drop(columns=['isPartial'])
df

⚠️ Retry 1 (2019-01-01): The request failed: Google returned a response with code 429


In [5]:
df_melted = df.melt(
    id_vars='date',
    var_name='keyword',
    value_name='interest'
)

df_melted['weekly_avg'] = (
    df_melted
    .groupby(['keyword', pd.Grouper(key='date', freq='W')])['interest']
    .transform('mean')
)

df_melted

,date,keyword,interest,weekly_avg
0,2019-01-01,gym,84,84.0
1,2019-02-01,gym,74,74.0
2,2019-03-01,gym,76,76.0
3,2019-04-01,gym,72,72.0
4,2019-05-01,gym,68,68.0
...,...,...,...,...
415,2025-08-01,quit smoking,1,1.0
416,2025-09-01,quit smoking,1,1.0
417,2025-10-01,quit smoking,1,1.0
418,2025-11-01,quit smoking,1,1.0


In [ ]:
df_melted['week'] = df_melted['date'].dt.isocalendar().week

ny = df_melted[df_melted['week'].between(1, 8)]

nyri = (
    ny
    .assign(period=lambda x: x['week'].apply(
        lambda w: 'early' if w <= 4 else 'late'
    ))
    .groupby(['keyword', 'period'])['interest']
    .mean()
    .unstack()
)

nyri['NYRI'] = nyri['late'] / nyri['early']
nyri = nyri.sort_values('NYRI', ascending=False)
